# 1회차 | 배열과 브로드캐스팅

**학습 목표**
- 리스트와 NumPy 배열 연산의 차이를 설명한다.
- 1차원·2차원 배열의 shape과 axis를 읽는다.
- 스칼라·행·열 브로드캐스팅을 적용한다.
- 호환되지 않는 shape을 실행 전에 판별한다.

> 실행 순서: 위에서 아래로 `Shift+Enter`를 누르세요. 오류가 나면 먼저 현재 폴더가 `notebooks`인지 확인하세요. 데이터 경로는 `../data`입니다.

In [ ]:
# 공통 준비: 라이브러리와 데이터 경로
from pathlib import Path
import numpy as np
import pandas as pd
DATA = Path('../data')
print('데이터 폴더:', DATA.resolve())

## 0. 도입 문제 — 실행 전에 예상하기 (10분)

다음 두 결과가 같을지 먼저 예상하세요.

- 파이썬 리스트 `[1, 2, 3] * 2`
- NumPy 배열 `np.array([1, 2, 3]) * 2`

컴퓨터에서 **반복**과 **수치 계산**은 같은 기호를 써도 자료형에 따라 뜻이 달라집니다.

In [ ]:
python_list = [1, 2, 3]
numpy_array = np.array([1, 2, 3])
print('리스트 × 2 :', python_list * 2)
print('배열 × 2   :', numpy_array * 2)
print('배열 + 10  :', numpy_array + 10)

## 1. 배열의 모양 읽기 (15분)

`shape`은 배열을 계산하기 전에 반드시 확인해야 하는 정보입니다.

- `(3,)` : 값 3개인 1차원 배열
- `(2, 3)` : 2행 3열인 2차원 배열
- `ndim` : 축의 개수
- `size` : 전체 원소 개수

In [ ]:
one = np.array([10, 20, 30])
two = np.array([[10, 20, 30], [40, 50, 60]])
for name, arr in [('one', one), ('two', two)]:
    print(name, 'shape=', arr.shape, 'ndim=', arr.ndim, 'size=', arr.size)

In [ ]:
# 확인 실습: 아래 배열의 shape을 실행 전에 종이에 적으세요.
a = np.array([[1, 2], [3, 4], [5, 6]])
b = np.array([[[1, 2, 3]], [[4, 5, 6]]])
print('a:', a.shape)
print('b:', b.shape)

## 2. 브로드캐스팅 규칙 (15분)

NumPy는 두 배열의 **뒤쪽 차원부터** 비교합니다. 각 차원이 다음 중 하나면 계산할 수 있습니다.

1. 두 크기가 같다.
2. 둘 중 하나의 크기가 1이다.
3. 한쪽에 해당 차원이 없다.

예: `(2, 3)`과 `(3,)`은 가능하지만 `(2, 3)`과 `(2,)`는 불가능합니다.

In [ ]:
table = np.array([[10, 20, 30], [40, 50, 60]])
scalar = 10
column_rule = np.array([1, 2, 3])
print('원본 shape:', table.shape)
print('스칼라 +10\n', table + scalar)
print('각 열에 [1,2,3] 더하기\n', table + column_rule)

### 손으로 먼저 계산하기

`[[10,20,30],[40,50,60]] + [1,2,3]`에서 `[1,2,3]`이 두 행에 반복된다고 생각하고 결과 6개를 직접 적으세요. 그런 다음 위 코드를 실행해 비교합니다.

## 3. 행별 값은 왜 reshape가 필요한가? (10분)

학생 2명의 점수 3과목에 학생별 가산점 5점, 10점을 주려 합니다. `(2,)`는 `(2,3)`의 마지막 축 3과 맞지 않습니다. `(2,1)`로 바꾸면 각 행을 따라 값이 확장됩니다.

In [ ]:
scores = np.array([[80, 90, 70], [75, 85, 95]])
student_bonus = np.array([5, 10])
print('변경 전:', student_bonus.shape)
student_bonus = student_bonus.reshape(2, 1)
print('변경 후:', student_bonus.shape)
print(scores + student_bonus)

In [ ]:
# 같은 뜻의 다른 작성법: np.newaxis
student_bonus2 = np.array([5, 10])[:, np.newaxis]
print(student_bonus2)
print('shape:', student_bonus2.shape)

## 4. 오류를 읽는 연습 (10분)

아래 코드는 일부러 오류가 나도록 작성했습니다. 오류가 났다는 사실보다 **어떤 두 shape이 충돌했는지** 찾는 것이 목표입니다.

In [ ]:
bad_bonus = np.array([5, 10])
try:
    scores + bad_bonus
except ValueError as e:
    print('예상된 오류:', e)
    print('scores shape =', scores.shape)
    print('bad_bonus shape =', bad_bonus.shape)

In [ ]:
# 호환 여부 판별 퀴즈: 실행 전에 True/False를 예상하세요.
pairs = [((2, 3), (3,)), ((2, 3), (2,)), ((4, 1), (1, 5)), ((3, 1, 5), (1, 4, 1))]
for left, right in pairs:
    try:
        result_shape = np.broadcast_shapes(left, right)
        print(left, '+', right, '가능 →', result_shape)
    except ValueError:
        print(left, '+', right, '불가능')

## 5. 적용 실습 — 성적 보정 시스템 (15분)

학생 4명, 과목 3개의 점수가 있습니다.

1. 과목별 가산점 `[2, 0, 3]`을 적용합니다.
2. 학생별 태도점수 `[1, 0, 2, 1]`을 모든 과목에 적용합니다.
3. 최종 점수는 100점을 넘지 않도록 제한합니다.
4. 학생별 평균을 구합니다.

In [ ]:
names = np.array(['민준', '서연', '지우', '도윤'])
scores = np.array([[80,90,70], [75,85,95], [98,88,96], [60,72,68]])
subject_bonus = np.array([2, 0, 3])
attitude_bonus = np.array([1, 0, 2, 1]).reshape(-1, 1)

final_scores = np.clip(scores + subject_bonus + attitude_bonus, 0, 100)
averages = final_scores.mean(axis=1)
print(final_scores)
for name, avg in zip(names, averages):
    print(f'{name}: 평균 {avg:.1f}')

In [ ]:
# 개별 실습: 평균 85점 이상 학생만 출력하세요.
mask = averages >= 85
print(names[mask])

# 도전: 과목별 평균(axis=0)도 구해 보세요.
subject_means = final_scores.mean(axis=0)
print('과목별 평균:', subject_means.round(1))

## 6. 마무리 점검 (5분)

1. `(3, 4)`와 `(4,)`는 계산 가능한가? 이유는?
2. `(3, 4)`에 행별 값 3개를 더하려면 어떤 shape이 필요한가?
3. `axis=0` 평균과 `axis=1` 평균은 각각 무엇을 뜻하는가?
4. 브로드캐스팅이 이미지 처리에서 유용한 이유를 예상해 보세요.

## 마무리 확인

- 오늘 사용한 핵심 메서드를 한 문장으로 설명해 보세요.
- 코드의 숫자나 조건을 바꾼 뒤 결과가 왜 달라졌는지 기록하세요.
- **도전:** 같은 개념을 연구회 개인 주제 데이터에 적용하세요.